In [1]:
import numpy as np
import pandas as pd

In [ ]:
# Data loading

print("Loading data...")

cols = ["age","workclass","fnlwgt","education","education-num",
        "marital-status","occupation","relationship","race","sex",
        "capital-gain","capital-loss","hours-per-week","native-country","income"]

data = pd.read_csv(
    "adult.data.csv",
    names=cols,
    na_values=" ?",
    skipinitialspace=True
)

Loading data...


In [ ]:
# Data Cleaning

data = data.dropna()

# Encode target variable
data["income"] = data["income"].apply(lambda x: 1 if x == ">50K" else 0)

# One-hot encode categorical features
data = pd.get_dummies(data)
data = data.astype(float)

X_raw = data.drop("income", axis=1).values
Y = data["income"].values.reshape(1, -1)

In [ ]:
# Activation functions

def relu(x):
    return np.maximum(0, x)

def drelu(x):
    return x > 0

def sigmoid(x):
    x = np.clip(x, -500, 500)      # prevents overflow
    return 1 / (1 + np.exp(-x))

In [ ]:
# Traning function

def train_model(X, Y, label):
    print("\n")
    print("Training with:", label)
    print(" ")

    X = X.T
    m = Y.shape[1]

    np.random.seed(1)
    w1 = np.random.randn(64, X.shape[0]) * np.sqrt(2 / X.shape[0])
    b1 = np.zeros((64, 1))

    w2 = np.random.randn(32, 64) * np.sqrt(2 / 64)
    b2 = np.zeros((32, 1))

    w3 = np.random.randn(1, 32) * np.sqrt(2 / 32)
    b3 = np.zeros((1, 1))

    lr = 0.01

    for i in range(1000):

   # Forward pass
        z1 = np.dot(w1, X) + b1
        a1 = relu(z1)

        z2 = np.dot(w2, a1) + b2
        a2 = relu(z2)

        z3 = np.dot(w3, a2) + b3
        a3 = sigmoid(z3)

     # Loss
        loss = -np.sum(
            Y * np.log(a3 + 1e-8) +
            (1 - Y) * np.log(1 - a3 + 1e-8)
        ) / m

  # Backpropagation
        dz3 = a3 - Y
        dw3 = np.dot(dz3, a2.T) / m
        db3 = np.sum(dz3, axis=1, keepdims=True) / m

        da2 = np.dot(w3.T, dz3)
        dz2 = da2 * drelu(z2)
        dw2 = np.dot(dz2, a1.T) / m
        db2 = np.sum(dz2, axis=1, keepdims=True) / m

        da1 = np.dot(w2.T, dz2)
        dz1 = da1 * drelu(z1)
        dw1 = np.dot(dz1, X.T) / m
        db1 = np.sum(dz1, axis=1, keepdims=True) / m

  # Update
        w1 -= lr * dw1
        b1 -= lr * db1
        w2 -= lr * dw2
        b2 -= lr * db2
        w3 -= lr * dw3
        b3 -= lr * db3

        if i % 100 == 0:
            print("Step:", i, "Loss:", loss)

            
    # Accuracy
    out = sigmoid(
        np.dot(
            w3,
            relu(np.dot(w2, relu(np.dot(w1, X) + b1)) + b2)
        ) + b3
    )

    pred = (out > 0.5).astype(int)
    acc = np.mean(pred == Y) * 100

    print("Final Accuracy:", acc, "%")

In [10]:
# EXP1: raw data

print("Training Model on Raw Data")
train_model(X_raw.copy(), Y, "RAW DATA")

Training Model on Raw Data


Training with: RAW DATA
 
Step: 0 Loss: 13.984804763738602
Step: 100 Loss: 0.639397043558746
Step: 200 Loss: 0.6074007975189283
Step: 300 Loss: 0.5876105735261693
Step: 400 Loss: 0.5751787287552629
Step: 500 Loss: 0.5672511812454095
Step: 600 Loss: 0.5621274380754766
Step: 700 Loss: 0.5587771198137638
Step: 800 Loss: 0.5565647154477438
Step: 900 Loss: 0.5550916150761225
Final Accuracy: 75.91904425539757 %


In [ ]:
# EXP2: min-max scaled data

print("\nPreparing Min-Max Scaled Data")

X_scaled = (X_raw - X_raw.min(axis=0)) / (
    X_raw.max(axis=0) - X_raw.min(axis=0) + 1e-8
)

print("Training model on Scaled Data")
train_model(X_scaled, Y, "min-max scaled data")


Preparing Min-Max Scaled Data
Training model on Scaled Data


Training with: min-max scaled data
 
Step: 0 Loss: 0.872477264588494
Step: 100 Loss: 0.535702255599113
Step: 200 Loss: 0.49102901525608383
Step: 300 Loss: 0.46325477623847616
Step: 400 Loss: 0.4426696964157702
Step: 500 Loss: 0.4279242481068865
Step: 600 Loss: 0.4171404568780066
Step: 700 Loss: 0.40883839400893823
Step: 800 Loss: 0.4022196213160535
Step: 900 Loss: 0.396684246237521
Final Accuracy: 82.21184853045054 %
